# RecVAE on fMRI — driver notebook

This notebook is a thin driver around the `recvae` package. All model and
training logic lives in `recvae/`; this notebook just wires it up and runs
the experiment end to end.

If you change a hyperparameter, edit `recvae/config.py` (or pass overrides
to `Config(...)` here).

## 1. Imports and setup

In [ ]:
import os
import sys

# Allow importing the package when this notebook is run from notebooks/
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import torch
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams["figure.facecolor"] = "#ffffff"
%matplotlib inline

from recvae import (
    Config,
    RecVAEModel,
    FMRIDataset,
    DeviceDataLoader,
    build_dataloader,
    fit,
    evaluate,
    set_seed,
    get_default_device,
    list_subject_files,
    load_subject_volumes,
    normalize_per_subject,
)

set_seed(2022)
device = get_default_device()
print("device:", device)

## 2. Load and normalize data

Set the `DIR_CN` / `DIR_AD` environment variables (or edit the cell below) to
point at your local copies. **Do not commit absolute paths or subject IDs.**

In [ ]:
DIR_CN = os.environ.get("FMRI_DIR_CN", "/path/to/CN")
DIR_AD = os.environ.get("FMRI_DIR_AD", "/path/to/AD")

cn_files = list_subject_files(DIR_CN)
ad_files = list_subject_files(DIR_AD)
print(f"CN: {len(cn_files)} subjects | AD: {len(ad_files)} subjects")

In [ ]:
TOL_TIME = 120

cn = load_subject_volumes(DIR_CN, cn_files, tol_time=TOL_TIME)
ad = load_subject_volumes(DIR_AD, ad_files, tol_time=TOL_TIME)
volumes = torch.cat([cn, ad], dim=0)
labels = torch.tensor([0] * cn.shape[0] + [1] * ad.shape[0], dtype=torch.long)
print("volumes:", tuple(volumes.shape), "  labels:", labels.bincount().tolist())

volumes, max_values, min_values = normalize_per_subject(volumes)
print("normalized to:", (float(volumes.min()), float(volumes.max())))

## 3. Configure and build model

In [ ]:
cfg = Config(tol_time=TOL_TIME)
train_size = volumes.shape[0]

ds = FMRIDataset(volumes)
dl = build_dataloader(ds, cfg.batch_size, shuffle=True, seed=cfg.seed)
dl = DeviceDataLoader(dl, device)

model = RecVAEModel(train_size=train_size, cfg=cfg).to(device)
h0 = torch.zeros(1, cfg.latent_dim, device=device)
print(model)

## 4. Train

Defaults match the original notebook (SGD@1e-6, 500 epochs). Override
`epochs`, `lr`, or `rho` via keyword arguments to `fit()`.

In [ ]:
history = fit(model, dl, h0, cfg=cfg, log_every=10)

## 5. Save

`state_dict()` includes the encoder/decoder/inference weights, the
`z_vectors` Parameter, and the `F_mat` Buffer.

In [ ]:
os.makedirs("Recorded", exist_ok=True)
torch.save(model.state_dict(), os.path.join("Recorded", "recvae_state.pt"))
torch.save(history["train_loss_history"], os.path.join("Recorded", "train_loss_history.pt"))

## 6. Evaluate (reconstruction)

**Caveat:** the original notebook used the training set as the test set —
no held-out subjects. Numbers below are training-set reconstruction loss
and should not be reported as generalization performance. See the README's
"Known limitations" section for the recommended subject-level k-fold split.

In [ ]:
test_dl = DeviceDataLoader(
    build_dataloader(ds, batch_size=1, shuffle=False), device,
)

losses = []
for batch, idx in test_dl:
    h_batch = h0.expand(batch.size(0), -1)
    which = idx.long()
    xs, mus, hs, ghs = evaluate(model, batch, h_batch, which)
    denom = 2 * batch.size(0) * cfg.tol_time
    loss1 = sum((x - mu).pow(2).sum() for x, mu in zip(xs, mus)) / (cfg.sig_x ** 2) / denom
    loss2 = sum((h - gh).pow(2).sum() for h, gh in zip(hs, ghs)) / (cfg.sig_h ** 2) / denom
    losses.append((loss1.item(), loss2.item()))

for i, (l1, l2) in enumerate(losses):
    print(f"subject {i}: loss1={l1:.4f}  loss2={l2:.4f}")

## 7. Inspect a reconstruction

Visualize one subject's middle timepoint vs its reconstruction.

In [ ]:
def show_three_slices(vol3d, title=""):
    fig, axes = plt.subplots(1, 3, figsize=(9, 3))
    x1, x2, x3 = 40, 40, 40
    axes[0].imshow(vol3d[x1, :, :].T, cmap="gray", origin="lower")
    axes[1].imshow(vol3d[:, x2, :].T, cmap="gray", origin="lower")
    axes[2].imshow(vol3d[:, :, x3].T, cmap="gray", origin="lower")
    fig.suptitle(title)
    plt.show()

subject_idx = 0
t_show = 60
with torch.no_grad():
    one = volumes[subject_idx:subject_idx + 1].to(device)
    h_b = h0.expand(1, -1)
    _, mus, _, _ = evaluate(model, one, h_b, torch.tensor([subject_idx], device=device))
    recon = mus[t_show][0, 0].cpu()
    orig = one[0, 0, :, :, :, t_show].cpu()
show_three_slices(orig, title=f"original t={t_show}")
show_three_slices(recon, title=f"reconstruction t={t_show}")